In [24]:
!{sys.executable} -m pip install scikit-learn



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip3.13 install --upgrade pip


In [25]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer

In [26]:
df = pd.read_csv("data/credit_risk_dataset.csv")
df = df.drop_duplicates().copy()

In [27]:
print("\nMin values:\n", df.min(numeric_only=True))
print("\nMax values:\n", df.max(numeric_only=True))
print("\nMedian values:\n", df.median(numeric_only=True))


Min values:
 person_age                      20.00
person_income                 4000.00
person_emp_length                0.00
loan_amnt                      500.00
loan_int_rate                    5.42
loan_status                      0.00
loan_percent_income              0.00
cb_person_cred_hist_length       2.00
dtype: float64

Max values:
 person_age                        144.00
person_income                 6000000.00
person_emp_length                 123.00
loan_amnt                       35000.00
loan_int_rate                      23.22
loan_status                         1.00
loan_percent_income                 0.83
cb_person_cred_hist_length         30.00
dtype: float64

Median values:
 person_age                       26.00
person_income                 55000.00
person_emp_length                 4.00
loan_amnt                      8000.00
loan_int_rate                    10.99
loan_status                       0.00
loan_percent_income               0.15
cb_person_cred_hist_

In [28]:
summary = df.describe().T
summary["median"] = df.median(numeric_only=True)
summary["missing"] = df[summary.index].isnull().sum()
display(summary)

,count,mean,std,min,25%,50%,75%,max,median,missing
person_age,32416.0,27.747008,6.354100,20.00,23.00,26.00,30.00,144.00,26.00,0
person_income,32416.0,66091.640826,62015.580269,4000.00,38542.00,55000.00,79218.00,6000000.00,55000.00,0
person_emp_length,31529.0,4.790510,4.145490,0.00,2.00,4.00,7.00,123.00,4.00,887
loan_amnt,32416.0,9593.845632,6322.730241,500.00,5000.00,8000.00,12250.00,35000.00,8000.00,0
loan_int_rate,29321.0,11.017265,3.241680,5.42,7.90,10.99,13.47,23.22,10.99,3095
loan_status,32416.0,0.218688,0.413363,0.00,0.00,0.00,0.00,1.00,0.00,0
loan_percent_income,32416.0,0.170250,0.106812,0.00,0.09,0.15,0.23,0.83,0.15,0
cb_person_cred_hist_length,32416.0,5.811297,4.059030,2.00,3.00,4.00,8.00,30.00,4.00,0


In [29]:
X = df.drop("loan_status", axis=1)
y = df["loan_status"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [30]:
# Clean invalid rows
def clean_invalid_rows(df):
    df = df.copy()
    df = df[(df["person_age"] > 0) & (df["person_age"] < 100)]
    df = df[df["person_income"] > 0]
    df = df[df["loan_amnt"] > 0]
    df = df[(df["person_emp_length"].isna()) | ((df["person_emp_length"] >= 0) & (df["person_emp_length"] <= 50))]
    df = df[(df["person_emp_length"].isna()) | (df["person_emp_length"] <= df["person_age"])]
    df = df[(df["loan_int_rate"].isna()) | (df["loan_int_rate"] > 0)]
    return df

# Cleaned the training and test sets separately to preserve the independence of the test set and avoid data leakage
X_train = clean_invalid_rows(X_train)
y_train = y_train.loc[X_train.index]
X_test = clean_invalid_rows(X_test)
y_test = y_test.loc[X_test.index]

In [31]:
#Add engineered features

def add_engineered_features(df):
    df = df.copy()
    df["income_to_loan_ratio"] = df["person_income"] / df["loan_amnt"]
    df["emp_length_to_age_ratio"] = df["person_emp_length"] / df["person_age"]
    return df

X_train = add_engineered_features(X_train)
X_test = add_engineered_features(X_test)

In [32]:
#Check the last 10 rows of emp_length_to_age_ratio
X_train["emp_length_to_age_ratio"].sort_values(ascending=False).head(10)

32515    0.716981
31867    0.673913
32263    0.673913
31866    0.659574
30914    0.645833
30776    0.627907
32206    0.625000
30829    0.625000
31888    0.619048
31830    0.619048
Name: emp_length_to_age_ratio, dtype: float64

In [33]:
#Check last 10 rows of income_to_loan_ratio sorted by desc
X_train["income_to_loan_ratio"].sort_values(ascending=False).head(10)

18917    283.333333
30049    241.394556
31924    225.000000
27877    208.800000
31922    206.363636
29188    200.000000
31919    197.142857
32037    197.142857
29527    186.000000
238      183.000000
Name: income_to_loan_ratio, dtype: float64

In [34]:
# Scale

categorical_features = [
    "person_home_ownership",
    "loan_intent",
    "loan_grade",
    "cb_person_default_on_file"
]

log_features = ["person_income", "loan_amnt", "income_to_loan_ratio"]

numeric_features = [
    "person_age",
    "person_emp_length",
    "loan_int_rate",
    "loan_percent_income",
    "cb_person_cred_hist_length",
    "emp_length_to_age_ratio"
]

# Log transform compresses large values and makes the distribution more balanced
log_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("log", FunctionTransformer(np.log1p, validate=False, feature_names_out="one-to-one")),
    ("scaler", StandardScaler())
])

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("log_num", log_transformer, log_features),
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Train processed shape:", X_train_processed.shape)
print("Test processed shape:", X_test_processed.shape)
feature_names = preprocessor.get_feature_names_out()

Train processed shape: (25925, 28)
Test processed shape: (6484, 28)


In [35]:
feature_names = preprocessor.get_feature_names_out()
print(feature_names)

['log_num__person_income' 'log_num__loan_amnt'
 'log_num__income_to_loan_ratio' 'num__person_age'
 'num__person_emp_length' 'num__loan_int_rate' 'num__loan_percent_income'
 'num__cb_person_cred_hist_length' 'num__emp_length_to_age_ratio'
 'cat__person_home_ownership_MORTGAGE' 'cat__person_home_ownership_OTHER'
 'cat__person_home_ownership_OWN' 'cat__person_home_ownership_RENT'
 'cat__loan_intent_DEBTCONSOLIDATION' 'cat__loan_intent_EDUCATION'
 'cat__loan_intent_HOMEIMPROVEMENT' 'cat__loan_intent_MEDICAL'
 'cat__loan_intent_PERSONAL' 'cat__loan_intent_VENTURE'
 'cat__loan_grade_A' 'cat__loan_grade_B' 'cat__loan_grade_C'
 'cat__loan_grade_D' 'cat__loan_grade_E' 'cat__loan_grade_F'
 'cat__loan_grade_G' 'cat__cb_person_default_on_file_N'
 'cat__cb_person_default_on_file_Y']


In [36]:
X_train_processed.shape

(25925, 28)

In [38]:
X_train_df = pd.DataFrame(
    X_train_processed,
    columns=preprocessor.get_feature_names_out()
)

X_train_df.head()

,log_num__person_income,log_num__loan_amnt,log_num__income_to_loan_ratio,num__person_age,num__person_emp_length,num__loan_int_rate,num__loan_percent_income,num__cb_person_cred_hist_length,num__emp_length_to_age_ratio,cat__person_home_ownership_MORTGAGE,...,cat__loan_intent_VENTURE,cat__loan_grade_A,cat__loan_grade_B,cat__loan_grade_C,cat__loan_grade_D,cat__loan_grade_E,cat__loan_grade_F,cat__loan_grade_G,cat__cb_person_default_on_file_N,cat__cb_person_default_on_file_Y
0,0.556657,-0.126717,0.545885,-0.761653,-1.196758,-1.456558,-0.756555,-0.939157,-1.295170,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0.198312,1.668953,-1.434647,-0.437951,1.076280,0.556523,2.147706,-0.444608,1.406870,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
2,-0.076316,-0.344148,0.249362,0.533157,-1.196758,-1.456558,-0.569183,0.791765,-1.295170,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,2.513896,-0.344148,2.476395,-0.114248,1.833959,-1.261429,-1.318670,0.049941,2.040681,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,-0.046043,0.376396,-0.448008,0.209455,-1.196758,1.281752,0.180303,0.791765,-1.295170,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
